In [4]:
import pandas as pd
import os

# 1. Define the path
file_path = '../data/raw/GSE9476_series_matrix.txt.gz'

# 2. Load the data
# sep='\t': It is separated by tabs, not commas.
# comment='!': Ignore all the metadata lines at the top.
# index_col=0: Use the first column (Probe IDs) as the row names.
print("Loading data... this might take a second.")
df = pd.read_csv(file_path, sep='\t', comment='!', index_col=0)

# 3. Check the shape
print(f"Data Loaded Successfully!")
print(f"Dimensions: {df.shape}")
print("\nFirst 5 rows:")
display(df.head())


import gzip

# 1. Capture the labels from the file header
labels = []
with gzip.open('../data/raw/GSE9476_series_matrix.txt.gz', 'rt') as f:
    for line in f:
        # Find the exact line you discovered
        if "!Sample_source_name_ch1" in line:
            # Split by tab to get individual entries
            parts = line.strip().split('\t')
            # The first part is the header title (!Sample...), so skip it (parts[1:])
            labels = parts[1:]
            break

# 2. Clean up the text (remove quotes)
# We want to turn "Bone Marrow CD34..." into just "Healthy" or "AML"
clean_labels = []
for label in labels:
    # Remove the quotes
    text = label.replace('"', '')
    
    # Logic: If it mentions CD34, it's Healthy. Otherwise, it's AML.
    if "CD34" in text:
        clean_labels.append('Healthy')
    else:
        clean_labels.append('AML')

# 3. Check if the lengths match (Safety Check)
print(f"Count of Labels found: {len(clean_labels)}")
print(f"Count of Columns in df: {len(df.columns)}")

if len(clean_labels) == len(df.columns):
    print("MATCH! Attaching labels to data...")
    # Create a new row in the dataframe for these labels
    df.loc['diagnosis'] = clean_labels
else:
    print("ERROR: Mismatch. Don't proceed.")
    
    
# Next we need to flip the dataframe(Transpose it)
df_t = df.T

# 2. Check the new shape
print(f"Old Shape: {df.shape} (Genes x Patients)")
print(f"New Shape: {df_t.shape} (Patients x Genes)")

# 3. Look at it
display(df_t.head())


df_t.groupby('diagnosis').mean()

# At this point we have too many genes(roughly 22k) vs too few patients(about 100)
# We want to pick the ones that actually tell us something
# First we are going to try and use regular manual code techqiue instead of relying on scikit learn

# Lets pick a small slice out of the genes, if we find high correlation eg. 0.99 they are redundant

# Pick the first 5 cols
subset = df_t.iloc[: , 0:5]

# Manually calc the correlatiun(Dot Prod Logic)
correlation_map = subset.corr()

print("Correlation Matrix (How much genes 'overlap'):")
print(correlation_map)

# 1. Group by diagnosis and calculate the mean for EVERY gene
mean_diffs = df_t.groupby('diagnosis').mean()

# 2. Subtract 'Healthy' from 'AML' to see the 'GAP'
# We transpose it back to make it a list of genes
gap = (mean_diffs.loc['AML'] - mean_diffs.loc['Healthy']).abs()

# 3. Sort them to see which genes have the biggest difference
top_genes = gap.sort_values(ascending=False).head(10)

print("Top 10 Genes with the largest difference between Healthy and AML:")
print(top_genes)


# 1. Calculate the variance for every gene column (ignoring 'diagnosis')
variances = df_t.drop('diagnosis', axis=1).var()

# 2. Let's look at the distribution of variance
print("Descriptive stats of our gene variances:")
print(variances.describe())

# 3. Filter: Keep only genes in the top 10% of variance
threshold = variances.quantile(0.9)
high_var_genes = variances[variances > threshold].index

print(f"\nOriginal gene count: {len(variances)}")
print(f"High variance gene count: {len(high_var_genes)}")

# 4. Create a new "Filtered" DataFrame
df_filtered = df_t[list(high_var_genes) + ['diagnosis']]

Loading data... this might take a second.
Data Loaded Successfully!
Dimensions: (22283, 64)

First 5 rows:


,GSM239170,GSM239323,GSM239324,GSM239326,GSM239328,GSM239329,GSM239331,GSM239332,GSM239333,GSM239334,...,GSM240500,GSM240501,GSM240502,GSM240503,GSM240504,GSM240505,GSM240506,GSM240507,GSM240508,GSM240509
ID_REF,,,,,,,,,,,,,,,,,,,,,
1007_s_at,3.016704,3.285669,2.929483,2.922820,3.159503,3.163327,2.985901,3.122709,3.070948,3.078003,...,3.134267,3.207050,2.971385,2.843774,3.037604,3.048857,3.113882,3.033006,3.212747,3.448303
1053_at,7.977735,6.532514,6.388007,6.466680,6.432795,6.407322,6.426471,6.376394,6.469070,6.621342,...,6.550238,6.916009,7.155750,6.471640,6.508025,6.631739,6.528610,6.684813,6.556031,6.341901
117_at,4.207281,4.994966,4.401597,4.747115,4.830046,4.213762,4.884418,4.431888,4.849665,4.432967,...,4.518836,4.193237,4.061884,4.175654,4.115092,5.090686,4.456707,3.923634,4.258683,4.327858
121_at,7.256095,7.420807,6.999340,7.094489,7.024333,7.179290,7.159899,7.009978,6.830979,7.208625,...,7.550825,7.289547,6.936977,7.136469,7.134417,7.281586,7.109759,7.110187,7.094449,7.096146
1255_g_at,2.204955,2.331625,2.133305,2.183329,2.127783,2.269698,2.264706,2.253940,2.287924,2.233190,...,2.286511,2.242720,2.199011,2.099221,2.214287,2.198988,2.162079,2.223595,2.256448,2.240602


Count of Labels found: 64
Count of Columns in df: 64
MATCH! Attaching labels to data...
Old Shape: (22284, 64) (Genes x Patients)
New Shape: (64, 22284) (Patients x Genes)


ID_REF,1007_s_at,1053_at,117_at,121_at,1255_g_at,1294_at,1316_at,1320_at,1405_i_at,1431_at,...,AFFX-r2-Hs28SrRNA-M_at,AFFX-r2-P1-cre-3_at,AFFX-r2-P1-cre-5_at,AFFX-ThrX-3_at,AFFX-ThrX-5_at,AFFX-ThrX-M_at,AFFX-TrpnX-3_at,AFFX-TrpnX-5_at,AFFX-TrpnX-M_at,diagnosis
GSM239170,3.016704,7.977735,4.207281,7.256095,2.204955,7.284373,4.265793,2.694352,3.646303,3.040292,...,5.662607,14.228826,13.713684,2.308299,3.315925,2.931302,2.921329,2.310954,2.979276,Healthy
GSM239323,3.285669,6.532514,4.994966,7.420807,2.331625,6.983594,4.970684,2.916325,8.817892,3.384867,...,6.186479,15.042417,14.486783,2.388966,3.40424,3.057272,3.216571,2.408195,3.111186,AML
GSM239324,2.929483,6.388007,4.401597,6.99934,2.133305,6.863371,4.595545,2.57813,11.424879,2.958851,...,5.821644,13.984025,13.287354,2.273666,3.193683,2.819624,2.770506,2.266408,2.856575,AML
GSM239326,2.92282,6.46668,4.747115,7.094489,2.183329,6.865971,4.575545,2.659718,10.747381,3.047881,...,6.145527,14.128244,13.76009,2.280654,3.269302,2.860894,2.870072,2.29356,2.95689,AML
GSM239328,3.159503,6.432795,4.830046,7.024333,2.127783,7.219841,4.547958,2.567437,9.296043,2.878563,...,6.359857,14.182431,13.60005,2.23884,3.232732,2.841193,2.801787,2.259069,2.902642,AML


Correlation Matrix (How much genes 'overlap'):
ID_REF     1007_s_at   1053_at    117_at    121_at  1255_g_at
ID_REF                                                       
1007_s_at   1.000000 -0.080884  0.044242  0.498513   0.479268
1053_at    -0.080884  1.000000  0.017473  0.229638   0.014289
117_at      0.044242  0.017473  1.000000 -0.070208   0.181320
121_at      0.498513  0.229638 -0.070208  1.000000   0.603138
1255_g_at   0.479268  0.014289  0.181320  0.603138   1.000000
Top 10 Genes with the largest difference between Healthy and AML:
ID_REF
205984_at      6.341879
209949_at      5.936711
211571_s_at    5.625091
221731_x_at    5.606216
215646_s_at     5.53836
204620_s_at    5.294789
204122_at      5.197888
205237_at      5.158033
202833_s_at    5.127281
214039_s_at    4.971594
dtype: object
Descriptive stats of our gene variances:
count     22283.000000
unique    22283.000000
top           0.044615
freq          1.000000
dtype: float64

Original gene count: 22283
High variance ge

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Pick the top gene you found (205984_at)
top_gene = '205984_at'

# 2. Create a boxplot to see the difference visually
plt.figure(figsize=(8, 6))
sns.boxplot(x='diagnosis', y=top_gene, data=df_t)
plt.title(f'Expression of {top_gene}: Healthy vs AML')
plt.show()